# Study Planner Agent - Demo Notebook
CSE476 CA1 - T24 Study Planner Agent

This notebook shows the agent running end to end. The agent decides which tool to call,
looks at the result, and decides what to do next (plan -> act -> observe -> decide -> act/final answer).

If `GITHUB_TOKEN` is set in `.env`, the agent uses real LLM tool-calling (GitHub Models, free OpenAI-compatible endpoint) to decide
which tool to call. If no key is set, it falls back to a small rule-based decision function
so this notebook still runs offline - either way the tools, memory and planner underneath
are the same real code, nothing is hardcoded or faked.

Each trace line below follows: `Agent decision -> Tool call -> Tool result -> Next decision -> Final answer`


In [1]:
import memory
memory.clear_memory()  # start with a clean slate for the demo
import agent
print("running in", ("LLM mode (Groq)" if agent.USING_GROQ else "LLM mode (Azure Foundry)") if agent.USING_LLM else "fallback (rule-based) mode")

running in fallback (rule-based) mode


## Demo 1 - Add tasks, then build a schedule
Adding 3 tasks in one request. The agent should call `add_task` three times, then call
`build_schedule` to actually produce the plan.

In [2]:
_ = agent.run_agent(
    "add task DBMS assignment due 2026-08-27 and add task OS quiz due 2026-08-30 and add task ML lab report due 2026-08-29"
)

[agent decision] received request: "add task DBMS assignment due 2026-08-27 and add task OS quiz due 2026-08-30 and add task ML lab report due 2026-08-29" (no GITHUB_TOKEN set -> rule-based fallback brain)
[tool call] add_task('DBMS assignment', '2026-08-27')
[tool result] {'status': 'ok', 'message': "added 'DBMS assignment', due 2026-08-27", 'urgent': False, 'task_count': 1}
[tool call] add_task('OS quiz', '2026-08-30')
[tool result] {'status': 'ok', 'message': "added 'OS quiz', due 2026-08-30", 'urgent': False, 'task_count': 2}
[tool call] add_task('ML lab report', '2026-08-29')
[tool result] {'status': 'ok', 'message': "added 'ML lab report', due 2026-08-29", 'urgent': False, 'task_count': 3}
[next decision] building schedule from current memory
[tool call] build_schedule()
[tool result] {'status': 'ok', 'blocks': [{'date': '2026-08-20', 'task': 'DBMS assignment', 'hours': 2}, {'date': '2026-08-20', 'task': 'ML lab report', 'hours': 2}, {'date': '2026-08-21', 'task': 'DBMS assignmen

## Demo 2 - Memory persists across turns
Here we add one more task in a completely separate request (a new "turn"). To prove this
isn't just a python variable sitting in memory for this notebook session, we run it in a
**brand new process** via the shell - if the task list still has our earlier 3 tasks plus
this new one, memory is genuinely persisted (it's reading `study_memory.json` off disk).

In [3]:
_ = agent.run_agent("add task viva revision due 2026-09-02")

[agent decision] received request: "add task viva revision due 2026-09-02" (no GITHUB_TOKEN set -> rule-based fallback brain)
[tool call] add_task('viva revision', '2026-09-02')
[tool result] {'status': 'ok', 'message': "added 'viva revision', due 2026-09-02", 'urgent': False, 'task_count': 4}
[next decision] building schedule from current memory
[tool call] build_schedule()
[tool result] {'status': 'ok', 'blocks': [{'date': '2026-08-20', 'task': 'DBMS assignment', 'hours': 2}, {'date': '2026-08-20', 'task': 'ML lab report', 'hours': 2}, {'date': '2026-08-21', 'task': 'DBMS assignment', 'hours': 2}, {'date': '2026-08-21', 'task': 'ML lab report', 'hours': 2}, {'date': '2026-08-22', 'task': 'OS quiz', 'hours': 2}, {'date': '2026-08-22', 'task': 'viva revision', 'hours': 2}, {'date': '2026-08-23', 'task': 'OS quiz', 'hours': 2}, {'date': '2026-08-23', 'task': 'viva revision', 'hours': 2}], 'warnings': [], 'task_count': 4}
[final answer] study plan:
  2026-08-20: DBMS assignment - 2h
  20

In [4]:
# prove it's real persistence, not just an in-memory python variable:
# run a totally separate python process and check the task list it sees
!python3 -c "import memory; print(memory.load_tasks())" 

[{'name': 'DBMS assignment', 'due': '2026-08-27'}, {'name': 'OS quiz', 'due': '2026-08-30'}, {'name': 'ML lab report', 'due': '2026-08-29'}, {'name': 'viva revision', 'due': '2026-09-02'}]


## Demo 3 - New urgent task -> automatic re-plan
We already have a schedule built above. Now an urgent task shows up (due almost immediately).
The agent should detect it's urgent from the `add_task` tool result and automatically call
`build_schedule` again to re-plan around it, without being explicitly told to rebuild.

In [5]:
_ = agent.run_agent("add task urgent viva prep due 2026-08-21")

[agent decision] received request: "add task urgent viva prep due 2026-08-21" (no GITHUB_TOKEN set -> rule-based fallback brain)
[tool call] add_task('urgent viva prep', '2026-08-21')
[tool result] {'status': 'ok', 'message': "added 'urgent viva prep', due 2026-08-21", 'urgent': True, 'task_count': 5}
[next decision] new task is urgent -> re-planning full schedule
[tool call] build_schedule()
[tool result] {'status': 'ok', 'blocks': [{'date': '2026-08-20', 'task': 'urgent viva prep', 'hours': 2}, {'date': '2026-08-20', 'task': 'DBMS assignment', 'hours': 2}, {'date': '2026-08-21', 'task': 'DBMS assignment', 'hours': 2}, {'date': '2026-08-21', 'task': 'ML lab report', 'hours': 2}, {'date': '2026-08-22', 'task': 'ML lab report', 'hours': 2}, {'date': '2026-08-22', 'task': 'OS quiz', 'hours': 2}, {'date': '2026-08-23', 'task': 'OS quiz', 'hours': 2}, {'date': '2026-08-23', 'task': 'viva revision', 'hours': 2}, {'date': '2026-08-24', 'task': 'viva revision', 'hours': 2}], 'warnings': ["'ur

## Honest failure case - not enough time
This is what happens when there just isn't enough time before a deadline. Instead of pretending
everything fits, the planner schedules what it can and the agent reports a clear warning.

In [6]:
memory.clear_memory()  # reset for a clean failure demo
_ = agent.run_agent(
    "add task Task A due 2026-08-21 and add task Task B due 2026-08-21 and add task Task C due 2026-08-21"
)

[agent decision] received request: "add task Task A due 2026-08-21 and add task Task B due 2026-08-21 and add task Task C due 2026-08-21" (no GITHUB_TOKEN set -> rule-based fallback brain)
[tool call] add_task('Task A', '2026-08-21')
[tool result] {'status': 'ok', 'message': "added 'Task A', due 2026-08-21", 'urgent': True, 'task_count': 1}
[tool call] add_task('Task B', '2026-08-21')
[tool result] {'status': 'ok', 'message': "added 'Task B', due 2026-08-21", 'urgent': True, 'task_count': 2}
[tool call] add_task('Task C', '2026-08-21')
[tool result] {'status': 'ok', 'message': "added 'Task C', due 2026-08-21", 'urgent': True, 'task_count': 3}
[next decision] new task is urgent -> re-planning full schedule
[tool call] build_schedule()
[tool result] {'status': 'ok', 'blocks': [{'date': '2026-08-20', 'task': 'Task A', 'hours': 2}, {'date': '2026-08-20', 'task': 'Task B', 'hours': 2}], 'warnings': ["'Task A' only got 2/4 study hours before its deadline (2026-08-21) - not enough time, sched

All three tasks are due the same day, and our daily study capacity (4 hrs/day, shared across
tasks) can't fit everyone's full 4 hours. Notice `Task C` gets 0 hours and shows up in the
warnings - the agent doesn't fake a schedule that doesn't actually work.

## Bonus - other edge cases (invalid date, past date, duplicate, no tasks)
These are handled by `tools.py` directly, shown here quickly for completeness.

In [7]:
import tools

memory.clear_memory()
print("no tasks yet:", tools.build_schedule())
print()
print("bad date format:", tools.add_task("bad task", "30-08-2026"))
print()
print("past date:", tools.add_task("late task", "2020-01-01"))
print()
print("first add:", tools.add_task("Math HW", "2026-08-25"))
print("duplicate add:", tools.add_task("math hw", "2026-08-26"))

no tasks yet: {'status': 'empty', 'message': 'no tasks yet, add some first', 'blocks': [], 'warnings': ['no tasks in memory, nothing to schedule']}

bad date format: {'status': 'error', 'message': "'30-08-2026' is not a valid date (use YYYY-MM-DD)"}

past date: {'status': 'error', 'message': "'2020-01-01' is in the past, can't schedule study time for that"}

first add: {'status': 'ok', 'message': "added 'Math HW', due 2026-08-25", 'urgent': False, 'task_count': 1}
duplicate add: {'status': 'error', 'message': "'math hw' is already in your task list (due 2026-08-25)"}
